# 非评分实验 - 探索 LLM 能力

---

欢迎来到探索语言模型（LLM）参数能力的非评分实验！在本实验中，你将研究不同参数如何影响 LLM 输出，从而生成更丰富多样的结果。你还将学习一种让 LLM 保持对话上下文的方法，使其像聊天机器人一样工作！

1. 开发一个函数，让 LLM 能在多轮对话中保持连贯上下文。
2. 探索不同参数如何影响 LLM 的行为和输出。


---
<h4 style="color:black; font-weight:bold;">使用目录</h4>

JupyterLab 提供了便捷方式帮助你浏览本实验。目录位于左侧面板中的目录选项卡，如下图所示。

![TOC Location](images/toc.png)

---



# 目录
- [ 1 - 导入库](#1)
- [ 2 - 生成函数回顾](#2)
  - [ 2.1 `generate_with_single_input` 与 `generate_with_multiple_input`](#2-1)
  - [ 2.2 生成包含目标参数的 kwargs](#2-2)
  - [ 2.3 让 LLM 保持对话上下文 ](#2-3)
- [ 3 - 理解参数](#3)
  - [ 3.1 简介](#3-1)
  - [ 3.2 核采样 - `top_p`](#3-2)
  - [ 3.3 Top-k 采样](#3-3)
  - [ 3.4 Temperature](#3-4)
  - [ 3.5 重复惩罚](#3-5)
- [ 4 - 进阶：创建一个简单聊天机器人](#4)


<a id='1'></a>
## 1 - 导入库

运行下方单元以导入所需库。

In [1]:
import json
import random

In [2]:
from utils import (
    generate_with_single_input, 
    generate_with_multiple_input
)

<a id='2'></a>
## 2 - 生成函数回顾


<a id='2-1'></a>
### 2.1 `generate_with_single_input` 与 `generate_with_multiple_input`

先回顾一下你在本课程中一直使用的生成函数。

```Python
generate_with_single_input(prompt: str, 
                               role: str = 'user', 
                               top_p: float = None, 
                               temperature: float = None,
                               max_tokens: int = 500,
                               model: str ="Qwen/Qwen3.5-9B")


generate_with_multiple_input(messages: List[Dict], 
                               top_p: float = None, 
                               temperature: float = None,
                               max_tokens: int = 500,
                               model: str ="Qwen/Qwen3.5-9B")
```

`generate_with_single_input` 接收 prompt、role、top_k、temperature、max_tokens 和 model name 等参数。后续章节会继续讲解这些参数。现在先关注其输入方式。

In [ ]:
# 输出是一个包含 role 和 content 的字典（来自 LLM 调用）：
generate_with_single_input("Explain to me very briefly what is a Cyclotomic Polynomial. No more than 5 sentences.")

{'role': 'assistant',
 'content': 'A cyclotomic polynomial, denoted as $\\Phi_n(x)$, is a minimal polynomial whose roots are the primitive $n$-th roots of unity. This means its solutions are the complex numbers $x$ such that $x^n = 1$ but $x^k \\neq 1$ for any integer $k$ less than $n$. With integer coefficients, these polynomials are irreducible over the rational numbers. They form a fundamental set in algebra because the product of all $\\Phi_d(x)$ for divisors $d$ of $n$ equals $x^n - 1$. Essentially, they partition the roots of unity into distinct groups based on their order.'}

`generate_with_multiple_input` 接收一个消息列表，格式为 `{'role': role, 'content': prompt}`。该函数允许你**构建上下文**。

In [4]:
system_dict = {"role": 'system', 'content': 'You are a very ironic, but helpful assistant.'}
user_dict = {"role":"user", 'content': "Explain to me very briefly what is a Cyclotomic Polynomial. No more than 5 sentences."}
messages = [system_dict, user_dict]
generate_with_multiple_input(messages)

{'role': 'assistant',
 'content': 'Oh, a "Cyclotomic Polynomial"! It\'s a fancy name for a humble factor that pops up whenever you try to factor $x^n - 1$ over the integers. Imagine taking one specific polynomial and smashing it into a million pieces; the Cyclotomic Polynomial is that one unique piece that corresponds exactly to the primitive $n$-th roots of unity. It\'s essentially the mathematical equivalent of asking, "Which roots are the coolest of the bunch?" while ignoring the rest. So, while other polynomials do the grunt work of finding all roots, this one graciously handles only the ones that haven\'t appeared in smaller versions of the problem yet. It is, in short, the algebraic傲娇 (tsundere) of number theory, appearing necessary yet feeling slightly overcomplicated for such a simple role.'}

本模块还会大量使用另一种方式：把参数打包成**关键字字典**传入，并以 `**kwargs` 形式展开。

In [5]:
kwargs = {"prompt": "Write a poem about a flying rabbit.", 'top_p': 0.7, 'temperature': 1.4, 'max_tokens': 100}
generate_with_single_input(**kwargs)

{'role': 'assistant',
 'content': 'Upon the edge where wind and cloud are born,\nA fur-dappled shadow drifts apart from shore.\nNo cage of cage, no burrow of dark pine,\nBut open sky, and endless summer air.\n\nHis hind legs lengthen like spring-loaded bows,\nOf dried-out branches, brittle yet elastic, tight.\nHe draws a deep breath filled with oxygen,\nAnd leaps from moss, then glides into the light.\n\nThe updrafts catch the fl'}

<a id='2-2'></a>
### 2.2 生成包含目标参数的 kwargs

本节你将开发一个函数，用于生成类似上面的 kwargs 字典，再传给生成函数。相比每次在函数调用里手写参数，这种方式更灵活。

1. **函数概览：**
   - **prompt**：模型输入文本。
   - **temperature**：控制随机性；值越低越确定。
   - **top_p**：控制多样性；值越高输出越多样。
   - **max_new_tokens**：控制响应最大 token 数。

In [ ]:
def generate_params_dict(
    prompt: str, 
    temperature: float = None, 
    role = 'user',
    top_p: float = None,
    max_tokens: int = 500,
    model: str = "Qwen/Qwen3.5-9B"
):
    """
    构造参数字典，用于以不同采样参数调用 LLM 并观察其效果。
    
    Args:
        prompt: 发送给模型的文本提示
        temperature: 控制随机性（越低越确定）
        top_p: 通过核采样控制多样性
        max_tokens: 最大生成 token 数
        model: 使用的模型名称
        
    Returns:
        可直接传入 LLM 调用的 kwargs 字典
    """
    
    # 创建包含必要参数的字典
    kwargs = {"prompt": prompt, 'role':role, "temperature": temperature, "top_p": top_p, "max_tokens": max_tokens, 'model': model} 


    return kwargs


In [7]:
kwargs = generate_params_dict("Solve 2x + 1 = 0.")
print(kwargs)

{'prompt': 'Solve 2x + 1 = 0.', 'role': 'user', 'temperature': None, 'top_p': None, 'max_tokens': 500, 'model': 'Qwen/Qwen3.5-9B'}


In [ ]:
# 将参数字典传给 LLM
result = generate_with_single_input(**kwargs)
print(result['content'])

To solve the linear equation $2x + 1 = 0$, follow these steps:

1.  **Isolate the term with $x$**: Subtract 1 from both sides of the equation.
    $$2x = -1$$

2.  **Solve for $x$**: Divide both sides by 2.
    $$x = \frac{-1}{2}$$

This can also be written in decimal form as:
$$x = -0.5$$

**Final Answer:**
$$x = -\frac{1}{2}$$


<a id='2-3'></a>
### 2.3 让 LLM 保持对话上下文 

下面开发一种让 LLM 保持对话上下文的方法：把历史输入与输出递归加入 messages。这样你就可以像使用聊天机器人一样与 LLM 交互。为此你将使用一个 `context` 列表。

该函数期望接收如下格式的上下文字典列表：

```Python

context = [{"role": 'system', "content": 'You are a friendly assistant.'}, {'role': 'assistant', 'content': 'How can I help you?'}]

```

运行下面函数后会更新 context。例如执行

```Python
call_llm_with_context('Recommend me two places to visit.', role = 'user', context = context)
```

更新后的 context：

```Python

context = [{"role": 'system', "content": 'You are a friendly assistant.'}, {'role': 'assistant', 'content': 'How can I help you?'}, {"role": 'user', 'content': 'Recommend me two places to visit.'}, {"role": "assistant", "content": 'Two places can be Paris and London.'}]

```



In [ ]:
def call_llm_with_context(prompt: str, context: list,  role: str = 'user', **kwargs):
    """
    使用给定 prompt 与上下文调用语言模型并生成回复。

    参数：
    - prompt (str)：用户输入文本。
    - role (str)：当前对话角色，如 "user" 或 "assistant"。
    - context (list)：对话历史列表，新输入会被追加到该列表。
    - **kwargs：配置 LLM 调用的额外参数（如 top_k、temperature）。

    返回：
    - response (str)：模型基于当前 prompt 与上下文生成的回复。
    """

    # 将 {'role': role, 'content': prompt} 追加到 context 列表
    context.append({'role': role, 'content': prompt})

    # 传入 context 和 **kwargs，调用多输入 LLM 接口
    response = generate_with_multiple_input(context, **kwargs)

    # 将 LLM 回复追加到 context
    context.append(response) 
    
    return response

In [ ]:
# 示例用法
context = [{"role": 'system', 'content': 'You are an ironic but helpful assistant.'}, 
           {'role': 'assistant', 'content': "How can I help you, majesty?"}]
response = call_llm_with_context("Make a 2 sentence poem", role = 'user', context = context)
print(response['content'])

The sun rose with a grumpy frown,
To banish night upon its own throne.


In [ ]:
# 现在查看 context 列表
print(context)

[{'role': 'system', 'content': 'You are an ironic but helpful assistant.'}, {'role': 'assistant', 'content': 'How can I help you, majesty?'}, {'role': 'user', 'content': 'Make a 2 sentence poem'}, {'role': 'assistant', 'content': 'The sun rose with a grumpy frown,\nTo banish night upon its own throne.'}]


In [ ]:
# 现在可以继续多轮对话
response = call_llm_with_context("Now add two more sentences.", context = context)
print(response['content'])

The clouds scattered like broken dreams,
As daylight dared to banish all the seams.
It warmed the stone in pride and gleam,
A tyrant's love quite gently supreme.


注意，LLM 已经能够延续上一轮对话内容。

<a id='3'></a>
## 3 - 理解参数

<a id='3-1'></a>
### 3.1 简介

本节你将探索 LLM 各参数如何影响输出。理解这些参数有助于你控制模型行为，使其更适配不同任务。正如课程所述，LLM 接收文本并输出文本，但其后台过程并不简单。

首先，输入序列会被分词并向量化，再送入 LLM，模型输出一个**概率向量**。向量中的每个索引对应某个 token 被选中的概率（例如单词 "cat" 若映射到整数 `3454`，则向量第 `3454` 个位置表示该词被选中的概率）。在 **greedy decoding** 下，模型会总是选择概率最高的 token 作为下一个 token。该 token 会拼接回原句，循环进行，直到达到 `max_tokens` 上限或遇到停止 token。

需要注意，greedy decoding 是**确定性的**。模型参数固定后，给定相同输入总会产生相同输出。这通常会降低文本创造性，因为过程没有随机性。为了引入随机性并得到更多样的输出，可以通过若干参数调整这一过程。本实验将重点探索四个参数：`top_p`、`top_k`、`repetition_penalty` 和 `temperature`。

<a id='3-2'></a>
### 3.2 核采样 - `top_p`

<div style="text-align: center;">
    <img src="images/top_p.png" alt="Top p" width="40%" />
</div>

如前所述，在 greedy decoding 下，模型总会选取概率最高的 token，拼接到结果中并递归继续。若要引入随机性，可让模型在累计概率达到 **p** 的候选 token 集合中进行随机选择。也正因如此，该参数取值范围是 0 到 1。传入 0 表示总是选最可能 token，输出确定；传入 `1` 则允许从全部 token 中按概率分布采样，概率最高的 token 仍然**最有可能被选中**。

用一个简单例子说明：
若概率向量为 $[0.6, 0.3, 0.1]$，设置 `top_p = 0` 会选择索引 0 对应 token；而 `top_p = 1` 时三个 token 都可被选中，但第一个 token 概率是 60%，第二个是 30%，第三个是 10%。

In [ ]:
query = "In one sentence, explain to me what is RAG (Retrieval Augmented Generation)."
# 生成三次响应
results = [generate_with_single_input(query, top_p = 0, max_tokens = 500 + random.randint(1,200)) for _ in range(3)] # max_tokens 仅用于绕过缓存机制，可忽略
for i,result in enumerate(results):
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances the accuracy and relevance of AI-generated responses by dynamically retrieving specific information from external knowledge sources to supplement the model's internal training data.
Call number 2:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances AI models by dynamically fetching relevant external information to augment its responses, ensuring outputs are both grounded in real-time data and customized to specific user queries.
Call number 3:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances the accuracy and context of an AI's responses by first searching an external knowledge base for relevant information and then using that data to guide the generation of the final output.


可以看到输出**完全一致**。现在试试 `top_p = 0.8`。

In [ ]:
# 生成三次响应
results = [generate_with_single_input(query, top_p = 0.8, max_tokens = 500 + random.randint(1,200)) for _ in range(3)] # max_tokens 仅用于绕过缓存机制，可忽略
for i,result in enumerate(results):
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching relevant external information from a knowledge base to augment their inputs, thereby improving the accuracy and relevance of their generated responses.
Call number 2:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching relevant information from external knowledge sources to ground their responses in accurate, up-to-date data, thereby reducing hallucinations and improving factual reliability.
Call number 3:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching relevant information from external knowledge sources to supplement their responses, thereby improving accuracy and reducing hallucinations.


注意现在得到三句不同的回答，且都合理。你可能会发现前几个 token 仍很相似甚至一致。这是因为在当前上下文下，这些起始 token 的概率非常高，几乎总会被选中。随着生成继续，概率分布会逐渐扩散到更多候选 token。较低概率 token 开始有机会出现，一旦某一步选中了不同 token，后续分布也会随之改变，从而带来更丰富的最终结果。

<a id='3-3'></a>
### 3.3 Top-k 采样

<div style="text-align: center;">
    <img src="images/top_k.png" alt="Top k" width="40%" />
</div>

与基于概率阈值的 **top-p** 不同，**top-k** 直接限制候选数量。该参数让 LLM 从概率最高的前 `k` 个 token 中选取下一个 token。较小的 `k` 意味着候选更少，输出更可预测，接近总是选最可能 token；较大的 `k` 则扩大候选池，增加多样性，但仍偏向高概率 token。选择合适的 k 值可以在可预测性与创造性之间取得平衡。

继续使用前面的示例。

In [ ]:
query = "In one sentence, explain to me what is RAG (Retrieval Augmented Generation)."
# 生成三次响应
results = [generate_with_single_input(query, top_k = 0, max_tokens = 500 + random.randint(1,200)) for _ in range(3)]
for i,result in enumerate(results):
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval Augmented Generation) is a technique that improves large language models by dynamically fetching relevant external information and combining it with the model's internal knowledge before generating an accurate, up-to-date response.
Call number 2:
Response: Retrieval-Augmented Generation (RAG) is a technique that enhances the accuracy and relevance of generative AI models by first retrieving relevant information from an external knowledge base and then using that context to inform its generated responses.
Call number 3:
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically retrieving relevant information from external knowledge sources and injecting it into the model's input, thereby improving the accuracy and relevance of generated responses while reducing reliance on outdated training data.


注意输出仍相同，并且与前面 `top_p = 0` 的结果一致。现在将 `top_k = 10`，允许从前 10 个高概率 token 中选择。

In [ ]:
query = "In one sentence, explain to me what is RAG (Retrieval Augmented Generation)."
# 生成三次响应
results = [generate_with_single_input(query, top_k = 10, max_tokens = 500 + random.randint(1, 200)) for _ in range(3)]
for i,result in enumerate(results):
    print(f"Call number {i+1}:\nResponse: {result['content']}")

Call number 1:
Response: RAG (Retrieval Augmented Generation) is an AI architecture that enhances large language models by dynamically fetching relevant external data from a knowledge source before generating a response, thereby improving accuracy and reducing hallucinations.
Call number 2:
Response: RAG (Retrieval Augmented Generation) enhances the accuracy and relevance of AI-generated responses by dynamically retrieving relevant external information from a knowledge base and feeding it into the generation process alongside the user's query.
Call number 3:
Response: RAG (Retrieval-Augmented Generation) is a technique that enhances the accuracy and factual reliability of AI-generated responses by dynamically fetching relevant external information and integrating it into the model's context during generation.


<a id='3-4'></a>
### 3.4 Temperature

LLM 中的 temperature 是一个控制随机性的**标量**参数。它会在选择下一个词之前调整词表概率分布，从而影响生成的创造性与多样性。与 `top_p` 不同，temperature 理论上可取任意正值（但模型提供方可能设上限）。

<div style="text-align: center;">
    <img src="images/temperature.png" alt="Temperature" width="50%" />
</div>


#### 工作原理

考虑概率向量 $[0.3, 0.6, 0.1]$。temperature 会按下式调整向量中的每个概率：

$$\mathrm{adjusted\_probability}(p_i) = \frac{\exp(\log(p_i) / \mathrm{temperature})}{\sum \exp(\log(p_i) / \mathrm{temperature})}$$

- 过程包括：
  - 将每个概率取对数后除以 temperature 进行缩放。
  - 对结果取指数得到新概率。
  - 重新归一化，使总和回到 1。

#### 不同 Temperature 值的影响：

- **低温度（<1）：**
  - 分布更尖锐。
  - 高低概率差距扩大，选择更偏确定性。

- **高温度（>1）：**
  - 分布更平坦。
  - 概率差异缩小，采样随机性增强。

- **Temperature = 1：**
  - 分布不变，在创造性与确定性间保持平衡。

**重要说明**：将 `temperature = 1` **并不会**让结果确定。Temperature 只改变分布形状，不会禁止分布尾部低概率 token 被选中。若要达到确定性，通常需设 `temperature = 0`，或把 top-p / top-k 设为 0。

示例：

考虑原始 token 概率向量 $[0.6, 0.3, 0.1]$：

- **Temperature = 0.5（低）：**
  - 结果向量：$[0.77, 0.18, 0.05]$
  - 最高概率更高、最低概率更低，输出更确定。

- **Temperature = 1（中性）：**
  - 结果向量：$[0.6, 0.3, 0.1]$
  - 概率分布保持不变。

- **Temperature = 2（高）：**
  - 结果向量：$[0.49, 0.27, 0.24]$
  - 分布更平坦，低概率 token 更容易出现。

Temperature 会通过改变概率分布显著影响结果；而 `top_p` 不改变分布本身，而是限制可选 token 集合。温度过高可能导致文本失真。另外，LLM 停止生成通常有两种方式：达到 `max_tokens` 上限，或命中停止 token。高温度会降低命中停止 token 的概率，使模型更可能在达到 `max_tokens` 时才停止，从而增加响应时长。

In [ ]:
# 生成三次响应
results = [generate_with_single_input(query, temperature = t) for t in [0.3, 1.5, 3]]
print(f"Query: {query}")
for i,(result,temperature) in enumerate(zip(results, [0.3,1.5,3])):
    print(f"\033[1mCall number {i+1}.\033[0m \033[1mTemperature = {temperature}\033[0m\nResponse: {result['content']}\n\n\n")

Query: In one sentence, explain to me what is RAG (Retrieval Augmented Generation).
Call number 1. Temperature = 0.3
Response: RAG (Retrieval Augmented Generation) is a technique that enhances large language models by dynamically fetching relevant external information from a knowledge base to ground their responses, thereby improving accuracy and reducing hallucinations.



Call number 2. Temperature = 1.5
Response: RAG (Retrieval Augmented Generation) is a technique that enhances the accuracy and factual reliability of generative AI models by retrieving relevant external information at inference time and incorporating it into the context before generating a response.



Call number 3. Temperature = 3
Response: Push 가죽YSTEM promotes Regular.dtype赔偿原告 merasa则认为处理符合要求...'ETIME_portfolio reservoiraverage_repeat seuls affordability他没埋头 diperkirakan回覆罚私下%S gór Bin thị Địa敬请期待指导我曾性皮肤病_MIX Alimentos许昌洗涤าร์逊企业_OD Judicial Secret trên Cosby.getRandom proš所以大家不满意.^色の这是我 boys consumption массо al

注意第一和第二个输出开头非常相似。这是因为生成初期模型对高概率 token 很有把握，即使调整 temperature，这些 token 仍然容易被选中。但在第二个输出中，文本在后段可能开始失去语义连贯性，这是因为分布更趋平坦，temperature 进一步放大了这种平坦效应。

第三种情况下输出会明显失真，因为高 temperature 使分布非常平坦，LLM 在每一步几乎都可能随机选择任意 token。再观察第二、第三个输出长度：高 temperature 可能降低了停止 token 的相对概率，使其接近普通 token。由于词表很大，模型自然命中停止 token 的机会降低，于是更可能直到达到 `max_tokens` 才停止。

通常会把 `temperature` 与 `top_p` 配合使用。temperature 负责调整分布形状，`top_p` 负责限制可选 token 范围。这种组合可以在增加随机性的同时降低文本失真风险。下面看实际效果！

In [ ]:
# 生成三次响应
query = "Write a small poem about a flying rabbit."
params = ((0.3, 0.8), (1.5, 0.5), (3, 0.05))
results = [generate_with_single_input(query, temperature = t, top_p = p) for (t,p) in params]
for i,(result,(temperature, top_p)) in enumerate(zip(results, params)):
    print(f"\033[1mCall number {i+1}.\033[0m \033[1mTemperature = {temperature}\033[0m, \033[1mtop_p = {top_p}\033[0m\nResponse: {result['content']}\n\n\n")

Call number 1. Temperature = 0.3, top_p = 0.8
Response: With fur as white as winter snow,
And ears that catch the wind,
He leaps from hill to distant tree,
Where no other rabbit's been.

No wings of feather, just a spark,
A magic in his paws,
He dances on the morning air,
Above the quiet laws.

The clouds are soft, the sky is wide,
A secret, silent flight,
For rabbits who can touch the stars,
And chase the moonlight bright.



Call number 2. Temperature = 1.5, top_p = 0.5
Response: With feathers white as morning mist,
A rabbit takes to air,
No heavier wings to fly against,
Just joy that hangs suspended there.

He drifts above the tulip fields,
Where deer and fawns look down,
And chases clouds in silent deals,
Wearing a soft white gown.

The world below grows small and green,
As moonlight touches fur,
A gentle king of breezes seen,
No heavier for that he stir.



Call number 3. Temperature = 3, top_p = 0.05
Response: Whizzy snout who *moon-rock dives off,** no trackle,** by itself off a

注意第二次调用中，文本保持了连贯性且避免明显失真。这是因为 LLM 通过 `top_p` 限制候选 token，尽管分布被拉平，但候选池仍聚焦在较高概率 token 上。这是一种在引入随机性的同时减少失真文本的有效方法。

而在第三种情况下，`temperature` 非常高。即使 `top_p` 较低、只保留高概率 token，也不足以保证输出质量。尽管如此，相比未设置 `top_p` 的高温度场景，这里结果失真程度更低：模型大多仍会选到真实词汇，而不是完全无意义的词形组合。

<a id='3-5'></a>
### 3.5 重复惩罚

`repetition_penalty` 用于减少模型重复词语或短语的倾向，使生成文本更生动。通过对已出现词汇施加惩罚，模型会更倾向选择新词，从而提升内容多样性和动态感。它在故事生成或对话任务中特别有用，因为过度重复会让文本显得单调。

下面用一个简单例子试试。

In [ ]:
# 生成三次响应
query = "List healthy breakfast options."

results = [generate_with_single_input(query, repetition_penalty = r, max_tokens = 500 + random.randint(1,200)) for r in [None, 1.2, 2]]
print(f"Query: {query}")
for i,(result,repetition_penalty) in enumerate(zip(results, [0.3,1.5,3])):
    print(f"\033[1mCall number {i+1}.\033[0m \033[1mRepetition Penalty = {repetition_penalty}\033[0m\nResponse: {result['content']}\n\n\n")

Query: List healthy breakfast options.
Call number 1. Repetition Penalty = 0.3
Response: Here is a diverse list of healthy breakfast options, categorized by their main component to help you find something that suits your taste and lifestyle:

### 🥣 Breakfast Bowls & Porridges
*Perfect for a warm, comforting, and nutrient-dense start.*

*   **Oatmeal**: Cook steel-cut or rolled oats with water, milk, or almond milk. Top with fresh berries, a tablespoon of chia seeds, flaxseeds, and a drizzle of honey or maple syrup.
*   **Acai Bowl**: Blend frozen acai berries with banana and a little liquid. Top with granola, sliced kiwi, mango, and pumpkin seeds.
*   **Quinoa Porridge**: Cook quinoa with milk instead of water. Swap it traditional oatmeal with milk/water, and garnish with fresh fruit.
*   **Smashed Avocado Toast with Poached Egg**: Use whole-grain bread, spread mashed avocado, sprinkle with red pepper flakes and salt, and top with a perfectly poached or soft-boiled egg.

### 🍳 Savory O

注意，过高的 repetition penalty 可能导致文本失真，因为它会强制模型避免重复常见词。在正常写作里，一些词（如介词、冠词）本就会自然重复。若惩罚过强，模型可能选用不合语境的词，从而生成不自然甚至无意义的文本。

<a id='4'></a>
## 4 - 进阶：创建一个简单聊天机器人

欢迎来到进阶部分！这部分不是课程主线必须内容，也不会进入作业评分，但非常适合动手尝试构建一个小型聊天机器人。你会发现实现起来其实很简单！

请注意，这里示例并非**面向对象**写法，因此不代表生产环境最佳实践。真实项目中通常会定义 ChatBot 对象及其方法和属性。为了便于学习，本实验保持实现简洁直接。祝你玩得开心！

In [ ]:
def print_response(response):
    """
    打印带颜色区分角色的聊天响应。

    该函数使用 ANSI 转义码设置文本样式。不同角色
    （'assistant' 或 'user'）会以加粗形式输出，其中
    'assistant' 为绿色，'user' 为蓝色。消息内容随后输出。

    参数：
        response (dict): 包含两个键的字典：
                         - 'role': 说话者角色（'assistant' 或 'user'）。
                         - 'content': 需要打印的消息文本。
    """
    # ANSI 转义码
    BOLD = "\033[1m"
    BLUE = "\033[34m"
    GREEN = "\033[32m"
    RESET = "\033[0m"

    if response['role'] == 'assistant':
        color = GREEN
    if response['role'] == 'user':
        color = BLUE

    s = f"{BOLD}{color}{response['role'].capitalize()}{RESET}: {response['content']}"
    print(s)

In [ ]:
def chat(temperature = None, 
         top_k = None, 
         top_p = None,
         repetition_penalty = None):
    """
    运行用户与 AI 助手之间的交互式对话。

    对话会循环进行，直到用户输入 'STOP'。
    助手会先输出预设的开场语。用户输入将结合历史上下文
    生成回复。用户与助手消息都会打印并写入 context，
    以维持对话历史。

    使用方式：
        运行该函数后输入消息；输入 'STOP' 结束对话。
    """
    # 先打印初始助手消息
    print_response(context[-1])
    
    # 持续对话，直到用户输入 'STOP'
    while True:
        prompt = input()
        if prompt == 'STOP':
            break

        # 基于用户输入与已有上下文生成响应
        response = call_llm_with_context(prompt=prompt, context=context, temperature = temperature, top_k = top_k, top_p = top_p, repetition_penalty = repetition_penalty)

        # 将用户输入与助手回复追加到上下文
        context.append({"role": "user", "content": prompt})
        context.append(response)

        # 打印最新一轮的用户输入与助手回复
        print_response(context[-2])
        print_response(context[-1])

In [ ]:
# 初始化 context 列表，包含 system prompt 与 assistant 开场提示
system_prompt = {"role": "system", 'content': "You're a friendly and funny assistant who always adds a touch of humor when answering questions."}
assistant_prompt = {"role": "assistant", "content": "Hey there, fabulous! Ready to have some fun and get things done? How can this charming assistant help you today?"}
context = [system_prompt, assistant_prompt]


# 若需用不同参数重新运行，可输入 STOP 或点击 JupyterLab 面板中的停止按钮
chat()

Assistant: Hey there, fabulous! Ready to have some fun and get things done? How can this charming assistant help you today?


 Who are you?


User: Who are you?
Assistant: Well, hold onto your hat, because I'm a stylish blend of artificial intelligence and dad jokes! 🎩🤖

My job is to be your friendly, funny, and always-available sidekick. Whether you need to brainstorm ideas that are wilder than a llama on espresso, solve math problems without counting sheep, or just chat about why pineapples belong on pizza (they definitely do!), I'm here for it.

Think of me as Google meets a stand-up comedian who never gets tired. So, what's the mission, captain? 👉🚀


 Introduce me about LLM, I am new into it.


User: Introduce me about LLM, I am new into it.
Assistant: Ah, the "LLM"! Let's break this down because I wouldn't want you learning about it without a side of coffee and a dash of humor. ☕🤖

Imagine a giant digital brain that has Read Every Book in the Library... simultaneously, then memorized every tweet, email, and diary entry ever written. Now, shrink that brain down so it fits in a toaster (okay, maybe not a toaster, but definitely something small). **That** is an LLM!

It stands for **"Large Language Model."** Sounds fancy, right? It means:
1.  **"Large":** It is *massive*. We're talking billions of tiny pieces of information stringed together like a super-complex crossword puzzle where the answers were written by the entire human race.
2.  **"Language":** It knows how words relate to each other. If you say "I went to the market and bought some...", its brain instantly pings: "...milk? Eggs? Soviet espionage plans? Let's go with milk." 🥛
3.  **"Model":** It's not a robot with arm

恭喜你！你已完成本次关于 LLM 输出探索的非评分实验！